In [1]:

import subprocess
from pathlib import Path

input_tif = Path("eth_ppt_2023_yravg_tc_100m_v1.tif")
output_dir = Path("compressed")
output_dir.mkdir(parents=True, exist_ok=True)

# Check whether the input file exists
if not input_tif.exists():
    raise FileNotFoundError(f"Input file not found: {input_tif.resolve()}")

compressions = {
    "DEFLATE": [
        "-co", "COMPRESS=DEFLATE",
        "-co", "PREDICTOR=3",
        "-co", "ZLEVEL=9",
        "-co", "TILED=YES",
    ],
    "ZSTD": [
        "-co", "COMPRESS=ZSTD",
        "-co", "PREDICTOR=3",
        "-co", "ZSTD_LEVEL=22",
        "-co", "TILED=YES",
    ],
    "LZW": [
        "-co", "COMPRESS=LZW",
        "-co", "PREDICTOR=3",
        "-co", "TILED=YES",
    ],
    "PACKBITS": [
        "-co", "COMPRESS=PACKBITS",
        "-co", "TILED=YES",
    ],
}

results = []

for name, options in compressions.items():
    output = output_dir / f"{input_tif.stem}_{name.lower()}.tif"

    cmd = [
        "gdal_translate",
        str(input_tif),
        str(output),
        "-of", "GTiff",
        *options,
    ]

    print(f"\nRunning {name}...")

    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)

        size_mb = output.stat().st_size / (1024 ** 2)
        results.append((name, size_mb, output))

        print(f"Success: {size_mb:.2f} MB")

    except subprocess.CalledProcessError as e:
        print(f"FAILED: {name}")
        print("GDAL error:")
        print(e.stderr or e.stdout or str(e))

    except OSError as e:
        print(f"Could not run GDAL: {e}")

print("\nResults:")
print("-" * 50)

for name, size, path in sorted(results, key=lambda x: x[1]):
    print(f"{name:12} {size:10.2f} MB  {path}")

if results:
    best = min(results, key=lambda x: x[1])
    print(f"\nSmallest: {best[0]} → {best[1]:.2f} MB")
else:
    print("\nAll compression attempts failed.")



Running DEFLATE...
Success: 60.17 MB

Running ZSTD...
Success: 50.89 MB

Running LZW...
Success: 129.35 MB

Running PACKBITS...
Success: 954.43 MB

Results:
--------------------------------------------------
ZSTD              50.89 MB  compressed/eth_ppt_2023_yravg_tc_100m_v1_zstd.tif
DEFLATE           60.17 MB  compressed/eth_ppt_2023_yravg_tc_100m_v1_deflate.tif
LZW              129.35 MB  compressed/eth_ppt_2023_yravg_tc_100m_v1_lzw.tif
PACKBITS         954.43 MB  compressed/eth_ppt_2023_yravg_tc_100m_v1_packbits.tif

Smallest: ZSTD → 50.89 MB
